In [ ]:
import json
import numpy as np
from transformers import RobertaTokenizerFast, RobertaForTokenClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import torch
from datasets import Dataset
from tqdm import tqdm
from seqeval.metrics import f1_score, precision_score, recall_score

In [ ]:
with open("./datasets/HateNorm/train.json", "r") as file:
    train_json = json.load(file)
with open("./datasets/HateNorm/valid.json", "r") as file:
    valid_json = json.load(file)
with open("./datasets/HateNorm/test.json", "r") as file:
    test_json = json.load(file)

In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained('roberta-large', add_prefix_space=True)

max_length_train = 0
max_length_valid = 0
max_length_test = 0
id2idx_train = {}
id2idx_valid = {}
id2idx_test = {}
for i in range(len(train_json)):
    tokenized = tokenizer(train_json[i]['sentence_tokens'], return_offsets_mapping=True, is_split_into_words=True, padding=False)
    train_json[i]['tokenized'] = tokenized
    max_length_train = max(len(tokenized['input_ids']), max_length_train)
    id2idx_train[str(train_json[i]['Id'])] = i
for i in range(len(valid_json)):
    tokenized = tokenizer(valid_json[i]['sentence_tokens'], return_offsets_mapping=True, is_split_into_words=True, padding=False)
    valid_json[i]['tokenized'] = tokenized
    max_length_valid = max(len(tokenized['input_ids']), max_length_valid)
    id2idx_valid[str(valid_json[i]['Id'])] = i
for i in range(len(test_json)):
    tokenized = tokenizer(test_json[i]['sentence_tokens'], return_offsets_mapping=True, is_split_into_words=True, padding=False)
    test_json[i]['tokenized'] = tokenized
    max_length_test = max(len(tokenized['input_ids']), max_length_test)

In [ ]:
train_json[id2idx_train[str(252)]]['tokenized']['offset_mapping'][33] = (1,0)
train_json[id2idx_train[str(252)]]['tokenized']['offset_mapping'][39] = (1,0)
train_json[id2idx_train[str(252)]]['tokenized']['offset_mapping'][41] = (1,0)
train_json[id2idx_train[str(273)]]['tokenized']['offset_mapping'][65] = (1,0)
train_json[id2idx_train[str(348)]]['tokenized']['offset_mapping'][19] = (1,0)
train_json[id2idx_train[str(522)]]['tokenized']['offset_mapping'][21] = (1,0)
train_json[id2idx_train[str(522)]]['tokenized']['offset_mapping'][22] = (1,0)
train_json[id2idx_train[str(645)]]['tokenized']['offset_mapping'][15] = (1,0)
train_json[id2idx_train[str(645)]]['tokenized']['offset_mapping'][31] = (1,0)
train_json[id2idx_train[str(645)]]['tokenized']['offset_mapping'][33] = (1,0)
train_json[id2idx_train[str(674)]]['tokenized']['offset_mapping'][2] = (1,0)
train_json[id2idx_train[str(674)]]['tokenized']['offset_mapping'][3] = (1,0)
train_json[id2idx_train[str(822)]]['tokenized']['offset_mapping'][32] = (1,0)
train_json[id2idx_train[str(826)]]['tokenized']['offset_mapping'][23] = (1,0)
train_json[id2idx_train[str(826)]]['tokenized']['offset_mapping'][29] = (1,0)
train_json[id2idx_train[str(826)]]['tokenized']['offset_mapping'][34] = (1,0)
train_json[id2idx_train[str(874)]]['tokenized']['offset_mapping'][17] = (1,0)
train_json[id2idx_train[str(883)]]['tokenized']['offset_mapping'][38] = (1,0)
train_json[id2idx_train[str(914)]]['tokenized']['offset_mapping'][26] = (1,0)
train_json[id2idx_train[str(919)]]['tokenized']['offset_mapping'][19] = (1,0)
valid_json[id2idx_valid[str(924)]]['tokenized']['offset_mapping'][8] = (1,0)
train_json[id2idx_train[str(1014)]]['tokenized']['offset_mapping'][25] = (1,0)
train_json[id2idx_train[str(1014)]]['tokenized']['offset_mapping'][28] = (1,0)
train_json[id2idx_train[str(1083)]]['tokenized']['offset_mapping'][23] = (1,0)
train_json[id2idx_train[str(1083)]]['tokenized']['offset_mapping'][26] = (1,0)
train_json[id2idx_train[str(1125)]]['tokenized']['offset_mapping'][9] = (1,0)
train_json[id2idx_train[str(1125)]]['tokenized']['offset_mapping'][21] = (1,0)
train_json[id2idx_train[str(1125)]]['tokenized']['offset_mapping'][33] = (1,0)
train_json[id2idx_train[str(1125)]]['tokenized']['offset_mapping'][47] = (1,0)
train_json[id2idx_train[str(1125)]]['tokenized']['offset_mapping'][51] = (1,0)
train_json[id2idx_train[str(1125)]]['tokenized']['offset_mapping'][52] = (1,0)
train_json[id2idx_train[str(1125)]]['tokenized']['offset_mapping'][54] = (1,0)
train_json[id2idx_train[str(1125)]]['tokenized']['offset_mapping'][58] = (1,0)
train_json[id2idx_train[str(1125)]]['tokenized']['offset_mapping'][60] = (1,0)
train_json[id2idx_train[str(1164)]]['tokenized']['offset_mapping'][13] = (1,0)
train_json[id2idx_train[str(1174)]]['tokenized']['offset_mapping'][19] = (1,0)
train_json[id2idx_train[str(1174)]]['tokenized']['offset_mapping'][24] = (1,0)
train_json[id2idx_train[str(1174)]]['tokenized']['offset_mapping'][26] = (1,0)
train_json[id2idx_train[str(1174)]]['tokenized']['offset_mapping'][30] = (1,0)
train_json[id2idx_train[str(1174)]]['tokenized']['offset_mapping'][34] = (1,0)
train_json[id2idx_train[str(1174)]]['tokenized']['offset_mapping'][33] = (1,0)
train_json[id2idx_train[str(1174)]]['tokenized']['offset_mapping'][37] = (1,0)
train_json[id2idx_train[str(1174)]]['tokenized']['offset_mapping'][38] = (1,0)
train_json[id2idx_train[str(1174)]]['tokenized']['offset_mapping'][42] = (1,0)
train_json[id2idx_train[str(1174)]]['tokenized']['offset_mapping'][43] = (1,0)
train_json[id2idx_train[str(1174)]]['tokenized']['offset_mapping'][45] = (1,0)
train_json[id2idx_train[str(1440)]]['tokenized']['offset_mapping'][8] = (1,0)
train_json[id2idx_train[str(1629)]]['tokenized']['offset_mapping'][25] = (1,0)
train_json[id2idx_train[str(1915)]]['tokenized']['offset_mapping'][28] = (1,0)
train_json[id2idx_train[str(1939)]]['tokenized']['offset_mapping'][26] = (1,0)
train_json[id2idx_train[str(1984)]]['tokenized']['offset_mapping'][10] = (1,0)
train_json[id2idx_train[str(2012)]]['tokenized']['offset_mapping'][8] = (1,0)
train_json[id2idx_train[str(2012)]]['tokenized']['offset_mapping'][14] = (1,0)
train_json[id2idx_train[str(2012)]]['tokenized']['offset_mapping'][16] = (1,0)
train_json[id2idx_train[str(2012)]]['tokenized']['offset_mapping'][22] = (1,0)
train_json[id2idx_train[str(2012)]]['tokenized']['offset_mapping'][24] = (1,0)
train_json[id2idx_train[str(2012)]]['tokenized']['offset_mapping'][30] = (1,0)
train_json[id2idx_train[str(2012)]]['tokenized']['offset_mapping'][40] = (1,0)
train_json[id2idx_train[str(2012)]]['tokenized']['offset_mapping'][43] = (1,0)
train_json[id2idx_train[str(2012)]]['tokenized']['offset_mapping'][53] = (1,0)
train_json[id2idx_train[str(2012)]]['tokenized']['offset_mapping'][56] = (1,0)
train_json[id2idx_train[str(2033)]]['tokenized']['offset_mapping'][13] = (1,0)
train_json[id2idx_train[str(2196)]]['tokenized']['offset_mapping'][9] = (1,0)
train_json[id2idx_train[str(2218)]]['tokenized']['offset_mapping'][8] = (1,0)
train_json[id2idx_train[str(2266)]]['tokenized']['offset_mapping'][5] = (1,0)
train_json[id2idx_train[str(2266)]]['tokenized']['offset_mapping'][12] = (1,0)
train_json[id2idx_train[str(2266)]]['tokenized']['offset_mapping'][15] = (1,0)
train_json[id2idx_train[str(2266)]]['tokenized']['offset_mapping'][27] = (1,0)
train_json[id2idx_train[str(2312)]]['tokenized']['offset_mapping'][19] = (1,0)
train_json[id2idx_train[str(2330)]]['tokenized']['offset_mapping'][16] = (1,0)
train_json[id2idx_train[str(2330)]]['tokenized']['offset_mapping'][34] = (1,0)
valid_json[id2idx_valid[str(2363)]]['tokenized']['offset_mapping'][7] = (1,0)
valid_json[id2idx_valid[str(2363)]]['tokenized']['offset_mapping'][23] = (1,0)
test_json[id2idx_test[str(56)]]['tokenized']['offset_mapping'][23] = (1,0)
test_json[id2idx_test[str(56)]]['tokenized']['offset_mapping'][27] = (1,0)
test_json[id2idx_test[str(56)]]['tokenized']['offset_mapping'][29] = (1,0)
test_json[id2idx_test[str(343)]]['tokenized']['offset_mapping'][25] = (1,0)
test_json[id2idx_test[str(343)]]['tokenized']['offset_mapping'][28] = (1,0)
test_json[id2idx_test[str(599)]]['tokenized']['offset_mapping'][13] = (1,0)
test_json[id2idx_test[str(599)]]['tokenized']['offset_mapping'][31] = (1,0)

In [ ]:
data_dict_train = {}
data_dict_valid = {}
data_dict_test = {}
for data in train_json:
    labels = [-100]
    tokenized = data['tokenized']
    att_labels = tokenized['attention_mask'].copy()
    i = -1
    att_labels[0] = 0
    for j in range(1, len(tokenized['offset_mapping'])):
        ofmap = tokenized['offset_mapping'][j]
        if ofmap==(0,0):
            labels.append(-100)
            att_labels[j] = 0
        elif ofmap[0]==0:
            i+=1
            labels.append(data['labels'][i])
        else:
            labels.append(-100)
            att_labels[j] = 0
    pad_list = [0 for p in range(max_length_train-len(labels))]
    pad_list_labels = [-100 for p in range(max_length_train-len(labels))]
    data_dict_train[str(data['Id'])] = {
        'input_ids':tokenized['input_ids']+pad_list, 
        'attention_mask':tokenized['attention_mask']+pad_list,
        'labels':labels+pad_list_labels, 
        'ID':data['Id'], 
        'offset_mapping':tokenized['offset_mapping'],
        'attention_labels':att_labels+pad_list,
        'raw_labels':data['labels']
    }

for data in valid_json:
    labels = [-100]
    tokenized = data['tokenized']
    att_labels = tokenized['attention_mask'].copy()
    i = -1
    att_labels[0] = 0
    for j in range(1, len(tokenized['offset_mapping'])):
        ofmap = tokenized['offset_mapping'][j]
        if ofmap==(0,0):
            labels.append(-100)
            att_labels[j] = 0
        elif ofmap[0]==0:
            i+=1
            labels.append(data['labels'][i])
        else:
            labels.append(-100)
            att_labels[j] = 0
    pad_list = [0 for p in range(max_length_valid-len(labels))]
    pad_list_labels = [-100 for p in range(max_length_valid-len(labels))]
    data_dict_valid[str(data['Id'])] = {
        'input_ids':tokenized['input_ids']+pad_list, 
        'attention_mask':tokenized['attention_mask']+pad_list,
        'labels':labels+pad_list_labels, 
        'ID':data['Id'], 
        'offset_mapping':tokenized['offset_mapping'], 
        'sentence_tokens':data['sentence_tokens'], 
        'labels_bio':data['labels_bio'], 
        'attention_labels':att_labels,
        'raw_labels':data['labels']
    }

for data in test_json:
    labels = [-100]
    tokenized = data['tokenized']
    att_labels = tokenized['attention_mask'].copy()
    i = -1
    att_labels[0] = 0
    for j in range(1, len(tokenized['offset_mapping'])):
        ofmap = tokenized['offset_mapping'][j]
        if ofmap==(0,0):
            labels.append(-100)
            att_labels[j] = 0
        elif ofmap[0]==0:
            i+=1
            labels.append(data['labels'][i])
        else:
            labels.append(-100)
            att_labels[j] = 0
    pad_list = [0 for p in range(max_length_test-len(labels))]
    pad_list_labels = [-100 for p in range(max_length_test-len(labels))]
    data_dict_test[str(data['Id'])] = {
        'input_ids':tokenized['input_ids']+pad_list, 
        'attention_mask':tokenized['attention_mask']+pad_list, 
        'labels':labels+pad_list_labels, 
        'ID':data['Id'], 
        'offset_mapping':tokenized['offset_mapping'], 
        'sentence_tokens':data['sentence_tokens'], 
        'labels_bio':data['labels_bio'], 
        'usage':data['usage'], 
        'attention_labels':att_labels,
        'raw_labels':data['labels']
    }
  

In [ ]:
data_list_train = list(data_dict_train.values())
data_list_valid = list(data_dict_valid.values())
data_list_test = list(data_dict_test.values())
dataset_train = Dataset.from_list(data_list_train)
dataset_valid = Dataset.from_list(data_list_valid)
dataset_test = Dataset.from_list(data_list_test)

In [ ]:
model = RobertaForTokenClassification.from_pretrained('roberta-large', num_labels=2)

args = TrainingArguments(
    output_dir="./results/RoBERTa/",
    per_device_train_batch_size=4,
    learning_rate=1e-6,
    num_train_epochs=20,
    logging_dir="./logs",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    include_for_metrics=["inputs"],
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset_train,
    eval_dataset=dataset_valid,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

trainer.train()
trainer.save_model("./results/RoBERTa/")

In [ ]:
model.eval()
model.to('cuda:0')
data_pred = []
for test_data in tqdm(data_list_test):
    input_ids_test = torch.tensor(test_data['input_ids']).unsqueeze(0).to('cuda:0')
    attention_mask_test = torch.tensor(test_data['attention_mask']).unsqueeze(0).to('cuda:0')
    with torch.no_grad():
        output = model(input_ids=input_ids_test, attention_mask=attention_mask_test)
    output_pred = output.logits.argmax(-1)[0].tolist()
    true_list = []
    pred_list = []
    pred_words = []
    list_index = 0
    for i in range(len(test_data['attention_labels'])):
        att = test_data['attention_labels'][i]
        ids = test_data['input_ids'][i]
        if ids==0 or ids==2:
            pass
        elif att==1:
            true_list.append(test_data['labels'][i])
            pred_list.append(output_pred[i])
            if output_pred[i]>0:
                pred_words.append(test_data['sentence_tokens'][list_index])
            list_index+=1
    acc = sum([int(a==b) for a,b in zip(true_list, pred_list)])/len(pred_list)
    data_pred.append({'ID':test_data['ID'], 'true':true_list, 'pred':pred_list, 'words':pred_words, 'accuracy':acc, 'usage':test_data['usage'], 'labels_bio':test_data['labels_bio']})

for i in range(len(data_pred)):
    pred_bio = []
    pre_p = 0
    for p in data_pred[i]['pred']:
        if p==0:
            pred_bio.append('O')
        elif p==1:
            if pre_p == 0:
                pred_bio.append('B')
            else:
                pred_bio.append('I')
        pre_p = p
    data_pred[i]['pred_bio'] = pred_bio
    if len(pred_bio)!=len(data_pred[i]['labels_bio']):
        data_pred[i]['is_error'] = True
    else:
        data_pred[i]['is_error'] = False

In [ ]:
#binary F1
cm_bi = [[0,0],[0,0]]
for data in data_pred:
    if not data['is_error']:
        for p,t in zip(data['pred'],data['true']):
            cm_bi[p][t] += 1
precision = cm_bi[1][1]/(cm_bi[1][1]+cm_bi[1][0])
recall = cm_bi[1][1]/(cm_bi[1][1]+cm_bi[0][1])
f1_bi = 2*precision*recall/(precision+recall)
print(f"binary F1: {f1_bi:.5f}")
print(f"binary precision: {precision:.5f}")
print(f"binary recall: {recall:.5f}")
#soft F1
cm_bio = [[0,0,0],[0,0,0],[0,0,0]]
bio2num = {'B':1,'I':2,'O':0}
for data in data_pred:
    if not data['is_error']:
        for p,t in zip(data['pred_bio'],data['labels_bio']):
            cm_bio[bio2num[p]][bio2num[t]] += 1
pre = (cm_bio[1][1]+cm_bio[2][2])/(sum(cm_bio[1])+sum(cm_bio[2]))
rec = (cm_bio[1][1]+cm_bio[2][2])/(cm_bio[0][1]+cm_bio[1][1]+cm_bio[2][1]+cm_bio[0][2]+cm_bio[1][2]+cm_bio[2][2])
f1_bio = 2*pre*rec/(pre+rec)
print(f"soft F1: {f1_bio:.5f}")
print(f"soft precision: {pre:.5f}")
print(f"soft recall: {rec:.5f}")
#hard F1
y_true = []
y_pred = []
for data in data_pred:
    if not data['is_error']:
        y_true.append(data['labels_bio'])
        y_pred.append(data['pred_bio'])
f1_span = f1_score(y_true, y_pred, average='macro')
pre_span = precision_score(y_true, y_pred, average='macro')
rec_span = recall_score(y_true, y_pred, average='macro')
print(f"hard F1: {f1_span:.5f}")
print(f"hard pre: {pre_span:.5f}")
print(f"hard rec: {rec_span:.5f}")